# Exploring the contents of HEASARC catalogs using Python

## Learning Goals

This notebook will teach you:
- How to retrieve and explore a HEASARC catalog's column names, descriptions, and units.
- How to retrieve the entire contents of a HEASARC catalog.
- How to retrieve a subset of a HEASARC catalog with easy-to-use Astroquery features.

## Introduction

This bite-sized tutorial will show you how to retrieve and explore the contents of HEASARC catalogs in Python.

To learn how to use Python to search for a particular HEASARC catalog, please see the '{doc}`Find specific HEASARC catalogs using Python <finding_relevant_heasarc_catalog>`' tutorial.

### Runtime

As of 5th March 2026, this notebook takes ~30 s to run to completion on Fornax using the 'small' server with 8GB RAM/ 2 cores.

## Imports

This notebook uses features from an Astroquery pre-release. You will need to install
the latest version using the command below. We will remove this once Astroquery
v0.4.12 is officially released.

```
pip install --pre astroquery --upgrade
```

In [1]:
from astroquery.heasarc import Heasarc

***

## 1. Listing a HEASARC catalog's columns

For this demonstration, we're assuming that you already have a HEASARC-hosted catalog
in mind; if not, you might find the
'{doc}`Find specific HEASARC catalogs using Python <finding_relevant_heasarc_catalog>`'
tutorial useful.

We will use the Archive of Chandra Cluster Entropy Profile Tables (ACCEPT) catalog
([Cavagnolo K. W. et al. 2009](https://ui.adsabs.harvard.edu/abs/2009ApJS..182...12C/abstract))
as an example.

The best way to get an idea of a catalog's contents is to list the column
names and descriptions. We can do this using the `Heasarc.list_columns(...)`
method, passing the name of the catalog as the first argument.

Each HEASARC catalog has a subset of 'standard' columns that will be returned by
default, which is why the table below contains only a few column names, descriptions, and
units even though this catalog has *79* columns:

In [2]:
# Pass the name of the catalog as the first argument
Heasarc.list_columns("acceptcat")

name,description,unit
str15,str80,str6
ra,Right Ascension,degree
name,Cluster Designation,
chi_squared_1,Chi-Squared of Best-Fit Model,
alpha_1,Best-Fit Power-Law Index,
num_radial_bins,Number of Radial Bins in Fit,
alpha_1_error,Uncertainty in Best-Fit Power-Law Index,
fit_method_1,Method of T_X Interpolation in Inner Region: flat or extr (Linear Extrapolation),
redshift,Redshift,
dof_1,Degrees of Freedom in Fit,


If, as is likely, you want to examine the full set of columns, you can pass the
`full=True` argument:

In [3]:
# Pass an additional argument to ensure all columns are returned
all_accept_cols = Heasarc.list_columns("acceptcat", full=True)
all_accept_cols

name,description,unit
str23,str80,str8
fit_method_4,Method of T_X Interpolation in Inner Region: flat or extr (Linear Extrapolation),
ra,Right Ascension,degree
bf_core_entropy_2,"Best-Fit Core Entropy, K_0",keV.cm^2
exposure_obsid_3,Exposure Time for Chandra Observation 3,s
name,Cluster Designation,
bii,Galactic Latitude,degree
bf_100k_entropy_4,Best-Fit Entropy at 100 kpc,keV.cm^2
exposure_obsid_2,Exposure Time for Chandra Observation 2,s
center_ccd_obsid_2,CCD Location of Cluster Center in Chandra Observation 2,


If you examine the output of the above cell, you'll notice that only part of the
table has been displayed; this is a common behavior when displaying long tables in
Jupyter notebooks, across multiple modules (e.g., Astropy, Pandas, etc.), as 'printing'
many lines can dramatically affect Jupyter's performance.

On the other hand, a table like this isn't going to destroy your computer, so we can
safely sidestep this issue by using the `pprint_all()` method of the `list_columns()`
output (an Astropy `Table` object, more on them in [Section 4](#4-interacting-with-heasarc-catalog-contents)):

In [4]:
# The 'pprint' stands for 'pretty print'
all_accept_cols.pprint_all()

          name                                            description                                      unit  
----------------------- -------------------------------------------------------------------------------- --------
           fit_method_4 Method of T_X Interpolation in Inner Region: flat or extr (Linear Extrapolation)         
                     ra                                                                  Right Ascension   degree
      bf_core_entropy_2                                                       Best-Fit Core Entropy, K_0 keV.cm^2
       exposure_obsid_3                                          Exposure Time for Chandra Observation 3        s
                   name                                                              Cluster Designation         
                    bii                                                                Galactic Latitude   degree
      bf_100k_entropy_4                                                      Best-Fit En

## 2. Retrieving the entire contents of a HEASARC catalog

The simplest use case of a HEASARC catalog is that you want to retrieve the
entire table.

We can easily fetch the entire catalog using Astroquery functions, but
before we do, we should check how many rows there are - we want to know what we're
getting into with respect to the size of the table.

Counting the rows in a HEASARC catalog involves writing a very simple 'Astronomical
Data Query Language' (ADQL) query.

ADQL is a cousin of the extremely popular 'Structured Query Language' (SQL) that has
been used for database management in industry for many years; the syntax is similar, but
with additions specific to astronomical searches.

We use the `COUNT(*)` function to return the number of rows in a table:

In [5]:
# Send query designed to count the rows of a catalog
accept_nrow_res = Heasarc.query_tap("SELECT COUNT(*) FROM acceptcat")

# Store the integer number of rows in a variable
accept_nrows = accept_nrow_res["count"][0]

# Visualize the returned table
accept_nrow_res

<DALResultsTable length=1>
count
int64
-----
  240

From the output above, we can see that there are 'only' 240 rows in the catalog; combine that information with
the number of columns (which we explored in [Section 1](#1-listing-a-heasarc-catalogs-columns)), and you
get a sense of the table's scale.

As the ACCEPT catalog is quite small (relatively speaking), we can retrieve the whole table without worrying
about download time or memory issues.

```{seealso}
A general tutorial on the many uses and features of ADQL is out of the scope of this
bite-sized demonstration. Various resources for learning ADQL are available online, such
as [this short course](https://docs.g-vo.org/adql/) ([Demleitner M. and Heinl H. 2024](https://dc.g-vo.org/voidoi/q/lp/custom/10.21938/uH0_xl5a6F7tKkXBSPnZxg)),
or the NASA Astronomical Virtual Observatories (NAVO)
[catalog queries tutorial](https://nasa-navo.github.io/navo-workshop/content/reference_notebooks/catalog_queries.html).
```

On the other hand, HEASARC hosts much larger catalogs than ACCEPT. The Chandra Source
Catalog 2 (CSC 2; [Evans I. N. et al. 2024](https://ui.adsabs.harvard.edu/abs/2024ApJS..274...22E/abstract)),
for instance:

In [6]:
# Same again, but CSC
Heasarc.query_tap("SELECT COUNT(*) FROM csc")

<DALResultsTable length=1>
count 
int64 
------
407806

```{warning}
For large catalogs like the CSC, we do not recommend retrieving the entire table at once.
```

Finally, now we know that retrieving the entire ACCEPT catalog is reasonable, we can
use the `query_region(...)` method of `Heasarc` to do just that. Few arguments are
required:
- `catalog="acceptcat"` - specifies the name of the catalog table to retrieve.
- `spatial="all-sky"` - overrides `query_region`'s default behavior of searching a catalog around a given coordinate, and instead considers the entire catalog.
- `columns="*"` - specifies that all columns should be returned (otherwise you will get a small subset, as discussed in [Section 1](#1-listing-a-heasarc-catalogs-columns).

In [7]:
accept_cat = Heasarc.query_region(catalog="acceptcat", spatial="all-sky", columns="*")
accept_cat

__row,name,obsid_1,ra,dec,lii,bii,exposure_obsid_1,center_ccd_obsid_1,redshift,kt,notes,obsid_2,exposure_obsid_2,center_ccd_obsid_2,obsid_3,exposure_obsid_3,center_ccd_obsid_3,obsid_4,exposure_obsid_4,center_ccd_obsid_4,obsid_5,exposure_obsid_5,center_ccd_obsid_5,obsid_6,exposure_obsid_6,center_ccd_obsid_6,obsid_7,exposure_obsid_7,center_ccd_obsid_7,fit_method_1,num_radial_bins,max_fit_radius,bf_core_entropy_1,bf_core_entropy_1_error,num_sigma_k0_gt_0_1,bf_100k_entropy_1,bf_100k_entropy_1_error,alpha_1,alpha_1_error,dof_1,chi_squared_1,p_value_1,fit_method_2,bf_core_entropy_2,bf_core_entropy_2_error,num_sigma_k0_gt_0_2,bf_100k_entropy_2,bf_100k_entropy_2_error,alpha_2,alpha_2_error,dof_2,chi_squared_2,p_value_2,fit_method_3,bf_core_entropy_3,bf_core_entropy_3_error,num_sigma_k0_gt_0_3,bf_100k_entropy_3,bf_100k_entropy_3_error,alpha_3,alpha_3_error,dof_3,chi_squared_3,p_value_3,fit_method_4,bf_core_entropy_4,bf_core_entropy_4_error,num_sigma_k0_gt_0_4,bf_100k_entropy_4,bf_100k_entropy_4_error,alpha_4,alpha_4_error,dof_4,chi_squared_4,p_value_4,__x_ra_dec,__y_ra_dec,__z_ra_dec
,,,deg,deg,deg,deg,s,,,keV,,,s,,,s,,,s,,,s,,,s,,,s,,,,Mpc,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,,
object,object,int32,float64,float64,float64,float64,float64,object,float64,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,object,int16,float64,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,float64,float64,float64
1,RX J1539.5-8335,8266,234.885354,-83.589953,307.565227,-22.291784,8000,I3,0.0728,4.29,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,29,0.20,21.8,3.10,7.1,115.1,5.8,1.32,0.11,26,13.29,9.81e-01,extr,0.0,--,--,135.3,4.5,0.83,0.04,27,40.39,4.71e-02,flat,25.9,2.90,9.1,110.0,5.8,1.41,0.12,26,13.52,9.79e-01,flat,0.0,--,--,133.7,4.5,0.79,0.04,27,54.08,1.49e-03,-0.0913244357272992,-0.0642187695832523,-0.993748357016153
2,Abell S0405,8272,57.886729,-82.219497,296.413320,-32.475665,7900,I3,0.0613,4.11,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,34,0.20,23.5,21.00,1.1,261.1,22.1,0.52,0.10,31,8.24,1.00e+00,extr,0.0,--,--,281.9,11.3,0.43,0.03,32,9.16,1.00e+00,flat,16.9,27.90,0.6,274.2,27.3,0.45,0.10,31,9.79,1.00e+00,flat,0.0,--,--,289.3,11.2,0.40,0.02,32,10.10,1.00e+00,0.114665363458883,0.0719664620044419,-0.990793965852339
3,Abell 3921,4973,342.490954,-64.428381,321.953076,-47.967314,29400,I3,0.0927,5.69,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,47,0.40,101.2,17.90,5.7,151.5,23.0,0.86,0.11,44,7.55,1.00e+00,extr,0.0,--,--,272.4,6.8,0.48,0.03,45,22.08,9.98e-01,flat,101.2,17.90,5.7,151.5,23.0,0.86,0.11,44,7.55,1.00e+00,flat,0.0,--,--,272.4,6.8,0.48,0.03,45,22.08,9.98e-01,-0.129861338063087,0.411640921482359,-0.902046442616797
4,Abell 3266,899,67.805433,-61.453497,272.147700,-40.152210,29800,I1,0.0590,9.07,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,15,0.08,63.7,41.90,1.5,405.3,51.6,0.71,0.27,12,0.79,1.00e+00,extr,0.0,--,--,418.9,37.8,0.44,0.06,13,2.02,1.00e+00,flat,72.5,49.70,1.5,376.7,48.0,0.64,0.28,12,1.26,1.00e+00,flat,0.0,--,--,404.6,35.2,0.39,0.05,13,2.34,1.00e+00,0.442464632176839,0.180517527128831,-0.878429548496581
5,Abell 3827,7920,330.471667,-59.945289,332.231736,-46.376682,45600,S3,0.0984,8.05,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,67,0.60,144.6,13.40,10.8,113.1,15.2,1.23,0.10,64,1651.91,6.60e-303,extr,0.0,--,--,287.2,7.4,0.60,0.03,65,4867.53,0.00e+00,flat,164.6,12.50,13.2,94.8,13.7,1.34,0.10,64,1368.56,6.59e-244,flat,0.0,--,--,293.5,7.3,0.57,0.03,65,5896.48,0.00e+00,-0.246834406824482,0.435775388415954,-0.865547564526947
6,Abell 3822,8269,328.5

## 3. Retrieving a subset of a HEASARC catalog

If you aren't interested in the _entire_ catalog, then we can also use Astroquery to
impose some restrictions on the rows we retrieve, based on the values of certain columns.

For example, perhaps we're only interested in galaxy clusters with a $z>0.4$. We saw in
[Section 1](#1-listing-a-heasarc-catalogs-columns) that the ACCEPT catalog includes
a column called `redshift`, we can use that to filter the rows we retrieve.

The `query_region(...)` call below will return all columns (`columns="*"`), and all rows where the
value of the `redshift` column is greater than 0.4 (`column_filters={"redshift": (">", "0.4")}`):

In [8]:
accept_cat_higherz = Heasarc.query_region(
    catalog="acceptcat",
    spatial="all-sky",
    column_filters={"redshift": (">", "0.4")},
    columns="*",
)
accept_cat_higherz

__row,name,obsid_1,ra,dec,lii,bii,exposure_obsid_1,center_ccd_obsid_1,redshift,kt,notes,obsid_2,exposure_obsid_2,center_ccd_obsid_2,obsid_3,exposure_obsid_3,center_ccd_obsid_3,obsid_4,exposure_obsid_4,center_ccd_obsid_4,obsid_5,exposure_obsid_5,center_ccd_obsid_5,obsid_6,exposure_obsid_6,center_ccd_obsid_6,obsid_7,exposure_obsid_7,center_ccd_obsid_7,fit_method_1,num_radial_bins,max_fit_radius,bf_core_entropy_1,bf_core_entropy_1_error,num_sigma_k0_gt_0_1,bf_100k_entropy_1,bf_100k_entropy_1_error,alpha_1,alpha_1_error,dof_1,chi_squared_1,p_value_1,fit_method_2,bf_core_entropy_2,bf_core_entropy_2_error,num_sigma_k0_gt_0_2,bf_100k_entropy_2,bf_100k_entropy_2_error,alpha_2,alpha_2_error,dof_2,chi_squared_2,p_value_2,fit_method_3,bf_core_entropy_3,bf_core_entropy_3_error,num_sigma_k0_gt_0_3,bf_100k_entropy_3,bf_100k_entropy_3_error,alpha_3,alpha_3_error,dof_3,chi_squared_3,p_value_3,fit_method_4,bf_core_entropy_4,bf_core_entropy_4_error,num_sigma_k0_gt_0_4,bf_100k_entropy_4,bf_100k_entropy_4_error,alpha_4,alpha_4_error,dof_4,chi_squared_4,p_value_4,__x_ra_dec,__y_ra_dec,__z_ra_dec
,,,deg,deg,deg,deg,s,,,keV,,,s,,,s,,,s,,,s,,,s,,,s,,,,Mpc,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,,
object,object,int32,float64,float64,float64,float64,float64,object,float64,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,object,int16,float64,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,float64,float64,float64
50,MACS J0257.1-2325,1654,44.288042,-23.434958,212.568244,-61.416916,19800,I3,0.5053,10.50,,3581,18500,I3,--,--,,--,--,,--,--,,--,--,,--,--,,extr,13,0.40,234.5,68.20,3.4,195.8,107.3,1.39,0.57,10,0.24,1.00e+00,extr,0.0,--,--,489.1,50.9,0.47,0.12,11,3.07,9.90e-01,flat,234.5,68.20,3.4,195.8,107.3,1.39,0.57,10,0.24,1.00e+00,flat,0.0,--,--,489.1,50.9,0.47,0.12,11,3.07,9.90e-01,0.640667436797526,0.656790501024463,-0.397707773611885
70,MACS J2214.9-1359,3259,333.739446,-14.002597,44.732830,-51.287190,19500,I3,0.5026,8.80,,5011,18500,I3,--,--,,--,--,,--,--,,--,--,,--,--,,extr,13,0.40,238.6,88.30,2.7,203.6,152.6,1.38,0.66,10,0.08,1.00e+00,extr,0.0,--,--,507.6,70.9,0.52,0.16,11,2.25,9.97e-01,flat,297.7,83.20,3.6,172.0,147.7,1.46,0.76,10,0.10,1.00e+00,flat,0.0,--,--,534.0,73.0,0.40,0.14,11,2.62,9.95e-01,-0.429306267142518,0.870142886221827,-0.241965878895571
81,MACS J0417.5-1154,3270,64.394525,-11.909086,205.946190,-39.478259,12000,I3,0.4400,11.07,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,11,0.30,9.5,6.70,1.4,101.6,14.8,1.52,0.22,8,0.88,9.99e-01,extr,0.0,--,--,117.2,9.2,1.29,0.13,9,2.51,9.81e-01,flat,27.1,7.30,3.7,99.7,15.1,1.42,0.23,8,1.16,9.97e-01,flat,0.0,--,--,136.1,9.4,0.85,0.08,9,7.22,6.14e-01,0.882381324716955,0.422869972297137,-0.206359357239186
82,RX J1347.5-1145,3592,206.877471,-11.752792,324.037353,48.807613,57700,I3,0.4510,10.88,,507,10000,S3,--,--,,--,--,,--,--,,--,--,,--,--,,extr,8,0.22,12.5,20.70,0.6,179.9,35.3,1.06,0.34,5,4.00,5.49e-01,extr,0.0,--,--,196.4,18.3,0.90,0.08,6,4.23,6.46e-01,flat,12.5,20.70,0.6,179.9,35.3,1.06,0.34,5,4.00,5.49e-01,flat,0.0,--,--,196.4,18.3,0.90,0.08,6,4.23,6.46e-01,-0.442606319178165,-0.873275588033894,-0.203689453746662
88,MACS J0159.8-0849,3265,29.956054,-8.833583,167.589086,-65.587449,17900,I3,0.4050,9.59,,6106,35300,I3,--,--,,--,--,,--,--,,--,--,,--,--,,extr,15,0.40,11.9,4.00,3.0,133.7,10.0,1.25,0.08,12,2.47,9.98e-01,extr,0.0,--,--,155.7,5.8,1.06,0.04,13,9.44,7.39e-01,flat,18.8,3.70,5.0,123.9,9.9,1.31,0.09,12,3.68,9.89e-01,flat,0.0,--,--,158.3,5.9,1.01,0.04,13,21.08,7.13e-02,0.49341276292893,0.8561317777342,-0.153565049840

If we want to further restrict the results, we can use boolean operators to add extra
filtering conditions. Here, for instance, we've decided we only want the
higher-redshift, low-central-entropy, galaxy clusters to be returned:

In [9]:
accept_cat_higherz_lowk = Heasarc.query_region(
    catalog="acceptcat",
    spatial="all-sky",
    column_filters={"redshift": (">", "0.4"), "bf_core_entropy_1": ("<", "15")},
    columns="*",
)
accept_cat_higherz_lowk

__row,name,obsid_1,ra,dec,lii,bii,exposure_obsid_1,center_ccd_obsid_1,redshift,kt,notes,obsid_2,exposure_obsid_2,center_ccd_obsid_2,obsid_3,exposure_obsid_3,center_ccd_obsid_3,obsid_4,exposure_obsid_4,center_ccd_obsid_4,obsid_5,exposure_obsid_5,center_ccd_obsid_5,obsid_6,exposure_obsid_6,center_ccd_obsid_6,obsid_7,exposure_obsid_7,center_ccd_obsid_7,fit_method_1,num_radial_bins,max_fit_radius,bf_core_entropy_1,bf_core_entropy_1_error,num_sigma_k0_gt_0_1,bf_100k_entropy_1,bf_100k_entropy_1_error,alpha_1,alpha_1_error,dof_1,chi_squared_1,p_value_1,fit_method_2,bf_core_entropy_2,bf_core_entropy_2_error,num_sigma_k0_gt_0_2,bf_100k_entropy_2,bf_100k_entropy_2_error,alpha_2,alpha_2_error,dof_2,chi_squared_2,p_value_2,fit_method_3,bf_core_entropy_3,bf_core_entropy_3_error,num_sigma_k0_gt_0_3,bf_100k_entropy_3,bf_100k_entropy_3_error,alpha_3,alpha_3_error,dof_3,chi_squared_3,p_value_3,fit_method_4,bf_core_entropy_4,bf_core_entropy_4_error,num_sigma_k0_gt_0_4,bf_100k_entropy_4,bf_100k_entropy_4_error,alpha_4,alpha_4_error,dof_4,chi_squared_4,p_value_4,__x_ra_dec,__y_ra_dec,__z_ra_dec
,,,deg,deg,deg,deg,s,,,keV,,,s,,,s,,,s,,,s,,,s,,,s,,,,Mpc,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,,
object,object,int32,float64,float64,float64,float64,float64,object,float64,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,object,int16,float64,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,float64,float64,float64
81,MACS J0417.5-1154,3270,64.394525,-11.909086,205.946190,-39.478259,12000,I3,0.4400,11.07,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,11,0.30,9.5,6.70,1.4,101.6,14.8,1.52,0.22,8,0.88,9.99e-01,extr,0.0,--,--,117.2,9.2,1.29,0.13,9,2.51,9.81e-01,flat,27.1,7.30,3.7,99.7,15.1,1.42,0.23,8,1.16,9.97e-01,flat,0.0,--,--,136.1,9.4,0.85,0.08,9,7.22,6.14e-01,0.882381324716955,0.422869972297137,-0.206359357239186
82,RX J1347.5-1145,3592,206.877471,-11.752792,324.037353,48.807613,57700,I3,0.4510,10.88,,507,10000,S3,--,--,,--,--,,--,--,,--,--,,--,--,,extr,8,0.22,12.5,20.70,0.6,179.9,35.3,1.06,0.34,5,4.00,5.49e-01,extr,0.0,--,--,196.4,18.3,0.90,0.08,6,4.23,6.46e-01,flat,12.5,20.70,0.6,179.9,35.3,1.06,0.34,5,4.00,5.49e-01,flat,0.0,--,--,196.4,18.3,0.90,0.08,6,4.23,6.46e-01,-0.442606319178165,-0.873275588033894,-0.203689453746662
88,MACS J0159.8-0849,3265,29.956054,-8.833583,167.589086,-65.587449,17900,I3,0.4050,9.59,,6106,35300,I3,--,--,,--,--,,--,--,,--,--,,--,--,,extr,15,0.40,11.9,4.00,3.0,133.7,10.0,1.25,0.08,12,2.47,9.98e-01,extr,0.0,--,--,155.7,5.8,1.06,0.04,13,9.44,7.39e-01,flat,18.8,3.70,5.0,123.9,9.9,1.31,0.09,12,3.68,9.89e-01,flat,0.0,--,--,158.3,5.9,1.01,0.04,13,21.08,7.13e-02,0.49341276292893,0.8561317777342,-0.15356504984051
101,MACS J0329.6-0211,3257,52.423671,-2.196575,186.444973,-44.674225,9900,I3,0.4500,5.20,,3582,19900,I3,6108,39600,I3,--,--,,--,--,,--,--,,--,--,,extr,14,0.40,6.6,2.70,2.4,102.9,6.5,1.21,0.07,11,9.63,5.64e-01,extr,0.0,--,--,115.4,3.6,1.08,0.03,12,14.83,2.51e-01,flat,11.1,2.50,4.4,96.7,6.4,1.26,0.07,11,11.91,3.71e-01,flat,0.0,--,--,117.5,3.6,1.03,0.03,12,26.77,8.33e-03,0.791959295208179,0.60936970170641,-0.0383280755531222
167,RX J1423.8+2404,1657,215.948996,24.077903,29.756425,68.985917,18500,I3,0.5450,5.92,,4195,115600,S3,--,--,,--,--,,--,--,,--,--,,--,--,,extr,7,0.22,10.2,5.00,2.0,119.9,10.8,1.27,0.17,4,1.75,7.82e-01,extr,0.0,--,--,133.8,7.3,1.02,0.05,5,15.01,1.03e-02,flat,10.2,5.00,2.0,119.9,10.8,1.27,0.17,4,1.75,7.82e-01,flat,0.0,--,--,133.8,7.3,1.02,0.05,5,15.01,1.03e-02,-0.535985261695602,-0.739103133782009,0.407978377954898
198,MACS J1621

## 4. Interacting with HEASARC catalog contents

The returns from our calls to the `Heasarc.query_region` method are Astropy
`Table` objects:

In [10]:
type(accept_cat)

astropy.table.table.Table

You can extract information similarly to a Pandas `DataFrame`; e.g., indexing with a
column name string retrieves the entries in that column:

In [11]:
# Extract source names from our subset of the ACCEPT catalog
accept_cat_higherz_lowk["name"]

MACS J0417.5-1154
RX J1347.5-1145
MACS J0159.8-0849
MACS J0329.6-0211
RX J1423.8+2404
MACS J1621.3+3810
3C 295


To retrieve the entries for a **row** in the table, you can index with an
integer; e.g., `0` for the first row:

In [12]:
accept_cat_higherz_lowk[0]

__row,name,obsid_1,ra,dec,lii,bii,exposure_obsid_1,center_ccd_obsid_1,redshift,kt,notes,obsid_2,exposure_obsid_2,center_ccd_obsid_2,obsid_3,exposure_obsid_3,center_ccd_obsid_3,obsid_4,exposure_obsid_4,center_ccd_obsid_4,obsid_5,exposure_obsid_5,center_ccd_obsid_5,obsid_6,exposure_obsid_6,center_ccd_obsid_6,obsid_7,exposure_obsid_7,center_ccd_obsid_7,fit_method_1,num_radial_bins,max_fit_radius,bf_core_entropy_1,bf_core_entropy_1_error,num_sigma_k0_gt_0_1,bf_100k_entropy_1,bf_100k_entropy_1_error,alpha_1,alpha_1_error,dof_1,chi_squared_1,p_value_1,fit_method_2,bf_core_entropy_2,bf_core_entropy_2_error,num_sigma_k0_gt_0_2,bf_100k_entropy_2,bf_100k_entropy_2_error,alpha_2,alpha_2_error,dof_2,chi_squared_2,p_value_2,fit_method_3,bf_core_entropy_3,bf_core_entropy_3_error,num_sigma_k0_gt_0_3,bf_100k_entropy_3,bf_100k_entropy_3_error,alpha_3,alpha_3_error,dof_3,chi_squared_3,p_value_3,fit_method_4,bf_core_entropy_4,bf_core_entropy_4_error,num_sigma_k0_gt_0_4,bf_100k_entropy_4,bf_100k_entropy_4_error,alpha_4,alpha_4_error,dof_4,chi_squared_4,p_value_4,__x_ra_dec,__y_ra_dec,__z_ra_dec
,,,deg,deg,deg,deg,s,,,keV,,,s,,,s,,,s,,,s,,,s,,,s,,,,Mpc,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,cm2 keV,cm2 keV,,cm2 keV,cm2 keV,,,,,,,,
object,object,int32,float64,float64,float64,float64,float64,object,float64,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,int32,float64,object,object,int16,float64,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,object,float64,float64,float64,float64,float64,float64,float64,int16,float64,object,float64,float64,float64
81,MACS J0417.5-1154,3270,64.394525,-11.909086,205.946190,-39.478259,12000,I3,0.4400,11.07,,--,--,,--,--,,--,--,,--,--,,--,--,,--,--,,extr,11,0.30,9.5,6.70,1.4,101.6,14.8,1.52,0.22,8,0.88,9.99e-01,extr,0.0,--,--,117.2,9.2,1.29,0.13,9,2.51,9.81e-01,flat,27.1,7.30,3.7,99.7,15.1,1.42,0.23,8,1.16,9.97e-01,flat,0.0,--,--,136.1,9.4,0.85,0.08,9,7.22,6.14e-01,0.882381324716955,0.422869972297137,-0.206359357239186


You can also convert the return to a Pandas `DataFrame` if you prefer working with
one of these data structures:

In [13]:
accept_cat_higherz_lowk_pd = accept_cat_higherz_lowk.to_pandas()
accept_cat_higherz_lowk_pd

,__row,name,obsid_1,ra,dec,lii,bii,exposure_obsid_1,center_ccd_obsid_1,redshift,...,bf_100k_entropy_4,bf_100k_entropy_4_error,alpha_4,alpha_4_error,dof_4,chi_squared_4,p_value_4,__x_ra_dec,__y_ra_dec,__z_ra_dec
0,81,MACS J0417.5-1154,3270,64.394525,-11.909086,205.946190,-39.478259,12000.0,I3,0.4400,...,136.1,9.4,0.85,0.08,9,7.22,6.14e-01,0.882381,0.422870,-0.206359
1,82,RX J1347.5-1145,3592,206.877471,-11.752792,324.037353,48.807613,57700.0,I3,0.4510,...,196.4,18.3,0.90,0.08,6,4.23,6.46e-01,-0.442606,-0.873276,-0.203689
2,88,MACS J0159.8-0849,3265,29.956054,-8.833583,167.589086,-65.587449,17900.0,I3,0.4050,...,158.3,5.9,1.01,0.04,13,21.08,7.13e-02,0.493413,0.856132,-0.153565
3,101,MACS J0329.6-0211,3257,52.423671,-2.196575,186.444973,-44.674225,9900.0,I3,0.4500,...,117.5,3.6,1.03,0.03,12,26.77,8.33e-03,0.791959,0.609370,-0.038328
4,167,RX J1423.8+2404,1657,215.948996,24.077903,29.756425,68.985917,18500.0,I3,0.5450,...,133.8,7.3,1.02,0.05,5,15.01,1.03e-02,-0.535985,-0.739103,0.407978
5,198,MACS J1621.3+3810,3254,245.353338,38.169069,60.940422,45.056261,9800.0,I3,0.4610,...,164.4,5.8,0.96,0.04,15,16.97,3.21e-01,-0.714566,-0.327858,0.617984
6,219,3C 295,2254,212.834500,52.202931,97.516619,60.802477,90900.0,I3,0.4641,...,109.3,3.8,1.18,0.04,15,34.84,2.59e-03,-0.332305,-0.514955,0.790186


## About this notebook

Author: David Turner, HEASARC Staff Scientist

Updated On: 2026-03-05

### Additional Resources

Support: [HEASARC Helpdesk](https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback?selected=heasarc)

[Latest Astroquery Documentation](https://astroquery.readthedocs.io/en/latest/)

[Short Course on ADQL Website](https://docs.g-vo.org/adql/)

[NAVO catalog queries tutorial](https://nasa-navo.github.io/navo-workshop/content/reference_notebooks/catalog_queries.html#using-the-tap-to-cross-correlate-and-combine)

[Latest PyVO Documentation](https://pyvo.readthedocs.io/en/latest/)

### Acknowledgements

### References

[Ginsburg, Sipőcz, Brasseur et al. (2019)](https://ui.adsabs.harvard.edu/abs/2019AJ....157...98G/abstract) - _astroquery: An Astronomical Web-querying Package in Python_

[Cavagnolo K. W., Donahue M., Voit G. M., Sun M. (2009)](https://ui.adsabs.harvard.edu/abs/2009ApJS..182...12C/abstract) - _Intracluster Medium Entropy Profiles for a Chandra Archival Sample of Galaxy Clusters_

[Evans I. N., Evans J. D., Martínez-Galarza J. R., Miller J. B. et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJS..274...22E/abstract) - _The Chandra Source Catalog Release 2 Series_

[Chandra Source Catalog 2 DOI - doi:10.25574/csc2](https://doi.org/10.25574/csc2)

[Demleitner M. and Heinl H. (2024)](https://dc.g-vo.org/voidoi/q/lp/custom/10.21938/uH0_xl5a6F7tKkXBSPnZxg) - _A Short Course on ADQL; Virtual Observatory Resource_